# Convolutional Neural Networks

Notebook for **Chapter 6** of the *dl* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/dl/ch06.html

Tip: change a number, predict the output, then run the cell again.

## 6.1 Convolution by Hand

In [ ]:
import numpy as np
img = np.array([[0,0,9,9,9]]*5, float)
k = np.array([[-1,0,1]]*3, float)
def conv2d(x, k, stride=1, pad=0):
    x = np.pad(x, pad); kh, kw = k.shape
    H = (x.shape[0]-kh)//stride + 1; W = (x.shape[1]-kw)//stride + 1
    out = np.zeros((H, W))
    for i in range(H):
        for j in range(W):
            out[i,j] = (x[i*stride:i*stride+kh, j*stride:j*stride+kw] * k).sum()
    return out
print(conv2d(img, k))
print(conv2d(img, k, pad=1).shape, conv2d(img, k, stride=2, pad=1).shape)

[[27. 27.  0.]
 [27. 27.  0.]
 [27. 27.  0.]]
(5, 5) (3, 3)


In [ ]:
print("conv 3x3x3->64", 3*3*3*64+64, "dense 224*224*3->1000", 224*224*3*1000+1000)

conv 3x3x3->64 1792 dense 224*224*3->1000 150529000


## 6.2 Pooling, Receptive Field, and a CNN in NumPy

In [ ]:
x = np.arange(16.).reshape(4,4)
print(x.reshape(2,2,2,2).max(axis=(1,3)))

[[ 5.  7.]
 [13. 15.]]


In [ ]:
layers = [(3,1),(3,1),(2,2),(3,1),(3,1),(2,2)]   # (kernel, stride)
rf = 1; jump = 1
for kk, s in layers:
    rf += (kk-1)*jump; jump *= s
    print("k", kk, "s", s, "RF", rf, "jump", jump)

k 3 s 1 RF 3 jump 1
k 3 s 1 RF 5 jump 1
k 2 s 2 RF 6 jump 2
k 3 s 1 RF 10 jump 2
k 3 s 1 RF 14 jump 2
k 2 s 2 RF 16 jump 4


In [ ]:
from numpy.lib.stride_tricks import sliding_window_view as swv
class CNN:
    def __init__(s, F=4, seed=0):
        r = np.random.default_rng(seed)
        s.K = r.normal(0, np.sqrt(2/9), (F,3,3)); s.kb = np.zeros(F)
        s.W = r.normal(0, np.sqrt(2/(F*9)), (F*9,3)); s.b = np.zeros(3)
    def forward(s, X):
        P = swv(X, (3,3), axis=(1,2)); s.P = P                       # (N,6,6,3,3) patches
        s.z = np.einsum('nijab,fab->nfij', P, s.K) + s.kb[None,:,None,None]
        s.a = np.maximum(0, s.z); N, F = s.a.shape[:2]
        s.pool = s.a.reshape(N,F,3,2,3,2); s.m = s.pool.max(axis=(3,5))
        s.flat = s.m.reshape(N,-1)
        l = s.flat @ s.W + s.b; e = np.exp(l - l.max(1, keepdims=True))
        s.p = e / e.sum(1, keepdims=True); return s.p
    def loss(s, X, y): return -np.log(s.forward(X)[np.arange(len(y)), y] + 1e-12).mean()
    def backward(s, y):
        N = len(y); d = s.p.copy(); d[np.arange(N), y] -= 1; d /= N
        gW = s.flat.T @ d; gb = d.sum(0); dm = (d @ s.W.T).reshape(s.m.shape)
        mask = (s.pool == s.m[:,:,:,None,:,None]); mask = mask / mask.sum(axis=(3,5), keepdims=True)
        da = (mask * dm[:,:,:,None,:,None]).reshape(s.a.shape); dz = da * (s.z > 0)
        gK = np.einsum('nijab,nfij->fab', s.P, dz); gkb = dz.sum(axis=(0,2,3))
        return gK, gkb, gW, gb

gradcheck K[1,0,2] numeric -0.01739227863950532 analytic -0.017392278625957425
0 1.1422
100 0.2905
200 0.2179
300 0.1662
CNN test acc 0.9633333333333334 params 151
dense test acc 0.9933333333333333 params 1091
--- train positions 0-3, test positions 4-7
CNN same-pos acc 1.0 new-pos acc 0.056666666666666664
dense same-pos acc 1.0 new-pos acc 0.0
global-pool CNN same-pos acc 1.0 new-pos acc 0.7233333333333334 params 81


## 6.3 CNN Architectures: From LeNet to ConvNeXt

In [ ]:
conv1 = 5*5*1*6+6; conv2 = 5*5*6*16+16; f1 = 400*120+120; f2 = 120*84+84; f3 = 84*10+10
print("LeNet5", conv1, conv2, f1, f2, f3, conv1+conv2+f1+f2+f3)

LeNet5 156 2416 48120 10164 850 61706


In [ ]:
std = 3*3*128*256; sep = 3*3*128 + 128*256; print("std", std, "sep", sep, "ratio", round(std/sep,2))

std 294912 sep 33920 ratio 8.69


## 6.4 1D and 3D Convolutions, and When to Choose a CNN

In [ ]:
print("1D out len:", 100-5+1, " 3D out:", (16+2-3)+1, (112+2-3)+1)

1D out len: 96  3D out: 16 112


In [ ]:
import torch.nn as nn
model = nn.Sequential(
    nn.Conv2d(1, 4, kernel_size=3), nn.ReLU(),
    nn.AdaptiveMaxPool2d(1), nn.Flatten(),     # global max pooling
    nn.Linear(4, 3))                            # 4*9+4 + 4*3+3 = 55 parameters

## 6.5 Looking Inside: Saliency, Grad-CAM, Integrated Gradients and Calibration

A trained CNN says "this is a dog" with 97 percent confidence. Two questions follow. First: *where in the image* did it look? Second: *should you believe the 97 percent*? The first is interpretability. The second is calibration. Both are standard in model reviews, and both are easy to do wrong, so we build each one by hand on a network small enough to hold in your head.

In [ ]:
import numpy as np
img = np.zeros((6, 6)); img[1:4, 3:6] = 1.0          # bright blob, upper right
k1 = np.ones((3, 3)) / 9.0                             # blob detector
k2 = np.array([[-1, 0, 1]] * 3) / 3.0                  # vertical edge detector
W = np.array([[2.0, 0.5], [-1.0, 1.0]])                # class 0 = "blob", class 1 = "other"

def conv(x, k):
    n = x.shape[0] - 2
    return np.array([[(x[i:i+3, j:j+3] * k).sum() for j in range(n)] for i in range(n)])

def forward(x):
    A = np.stack([np.maximum(conv(x, k1), 0), np.maximum(conv(x, k2), 0)])
    return A, W @ A.mean(axis=(1, 2))

A, logits = forward(img)
print("pooled features", np.round(A.mean(axis=(1, 2)), 3), "logits", np.round(logits, 3))

alpha = W[0] / 16                                      # d logit0 / d A_k = W[0,k] / (4*4)
cam = np.maximum((alpha[:, None, None] * A).sum(0), 0)
print("alpha", np.round(alpha, 4))
print("CAM / max")
print(np.round(cam / cam.max(), 2))

# occlusion: grey out each 2x2 patch and record the drop in the class-0 logit
base = logits[0]
drop = np.zeros((5, 5))
for i in range(5):
    for j in range(5):
        x = img.copy(); x[i:i+2, j:j+2] = 0.0
        drop[i, j] = base - forward(x)[1][0]
print("occlusion drop in logit 0")
print(np.round(drop, 3))

pooled features [0.333 0.333] logits [ 0.833 -0.   ]
alpha [0.125  0.0312]
CAM / max
[[0.   0.39 0.61 0.67]
 [0.   0.58 0.92 1.  ]
 [0.   0.39 0.61 0.67]
 [0.   0.19 0.31 0.33]]
occlusion drop in logit 0
[[0.    0.    0.083 0.16  0.104]
 [0.    0.    0.208 0.399 0.26 ]
 [0.    0.    0.25  0.479 0.312]
 [0.    0.    0.125 0.24  0.156]
 [0.    0.    0.    0.    0.   ]]


In [ ]:
import numpy as np
w = np.array([1.5, -2.0, 0.5]); b = 0.1
f = lambda x: 1 / (1 + np.exp(-(x @ w + b)))
x = np.array([1.0, 0.5, 2.0]); base = np.zeros(3)
grad = lambda p: f(p) * (1 - f(p)) * w                 # exact gradient of the sigmoid output
steps = 50
alphas = (np.arange(steps) + 0.5) / steps              # midpoint rule along the straight path
avg_grad = np.mean([grad(base + a * (x - base)) for a in alphas], axis=0)
ig = (x - base) * avg_grad
print("plain gradient   ", np.round(grad(x), 4))
print("integrated grads ", np.round(ig, 4))
print("sum of IG        ", round(ig.sum(), 4))
print("f(x) - f(base)   ", round(f(x) - f(base), 4))

plain gradient    [ 0.2096 -0.2795  0.0699]
integrated grads  [ 0.307  -0.2047  0.2047]
sum of IG         0.307
f(x) - f(base)    0.307


In [ ]:
import numpy as np
conf = np.array([0.95, 0.9, 0.92, 0.97, 0.85, 0.6, 0.65, 0.55, 0.7, 0.99])
hit  = np.array([1,    1,   0,    1,    0,    1,   0,    1,    0,   1])
edges = [0.5, 0.75, 1.0]
ece = 0.0
for lo, hi in zip(edges[:-1], edges[1:]):
    m = (conf > lo) & (conf <= hi)
    gap = abs(hit[m].mean() - conf[m].mean())
    ece += m.mean() * gap
    print(f"bin ({lo:.2f},{hi:.2f}] n={m.sum()} acc={hit[m].mean():.3f} conf={conf[m].mean():.3f} gap={gap:.3f}")
print("ECE", round(ece, 4))

bin (0.50,0.75] n=4 acc=0.500 conf=0.625 gap=0.125
bin (0.75,1.00] n=6 acc=0.667 conf=0.930 gap=0.263
ECE 0.208


In [ ]:
import numpy as np
rng = np.random.default_rng(0)
n = 4000
y = rng.integers(0, 3, n)
z = rng.normal(0, 1.0, (n, 3)); z[np.arange(n), y] += 1.6   # raw logits, honest signal
z = z * 3.0                                                  # simulate an overconfident network

def softmax(z, T=1.0):
    e = np.exp((z - z.max(1, keepdims=True)) / T)
    return e / e.sum(1, keepdims=True)

def nll(p): return -np.log(p[np.arange(n), y]).mean()

def ece(p, bins=10):
    conf, pred = p.max(1), p.argmax(1)
    out = 0.0
    for lo in np.linspace(0, 1, bins, endpoint=False):
        m = (conf > lo) & (conf <= lo + 1 / bins)
        if m.any(): out += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return out

Ts = np.arange(0.5, 6.01, 0.1)
best = Ts[np.argmin([nll(softmax(z, T)) for T in Ts])]
for T in (1.0, best):
    p = softmax(z, T)
    print(f"T={T:.1f} NLL={nll(p):.4f} ECE={ece(p):.4f} accuracy={(p.argmax(1)==y).mean():.3f}")

T=1.0 NLL=0.6015 ECE=0.0954 accuracy=0.787
T=1.9 NLL=0.5104 ECE=0.0095 accuracy=0.787


## Chapter recap

## Practice set

**1.** (easy) Output size for `n = 28`, `k = 5`, `p = 0`, `s = 1`. **Answer:** `28 − 5 + 1 = 24`.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/dl/ch06.html